# Vehicle Review Summary Generator
This notebook generates concise summaries using OpenAI GPT-4 from complete_review.json and updates final_data.json

In [1]:
# Import required libraries
import json
import os
import re
import time
from openai import OpenAI
from dotenv import load_dotenv
from typing import Dict, List

# Load environment variables
load_dotenv()

# Initialize OpenAI client
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

print("✓ Libraries imported successfully")
print("✓ OpenAI client initialized")

✓ Libraries imported successfully
✓ OpenAI client initialized


In [8]:
# Load data files
with open('complete_review.json', 'r', encoding='utf-8') as f:
    complete_reviews = json.load(f)

with open('final_data.json', 'r', encoding='utf-8') as f:
    final_data = json.load(f)

print(f"✓ Loaded {len(complete_reviews)} reviews from complete_review.json")
print(f"✓ Loaded {len(final_data)} records from final_data.json")

✓ Loaded 266 reviews from complete_review.json
✓ Loaded 1974 records from final_data.json


## Step 1: Generate model_id for each review

In [4]:
def generate_model_id(make_name: str, model_name: str) -> str:
    """
    Convert make_name and model_name to snake_case format.
    Example: 'Land Rover' + 'Discovery' -> 'land_rover_discovery'
    """
    # Convert to lowercase and replace spaces/special chars with underscores
    make_clean = re.sub(r'[^a-z0-9]+', '_', make_name.lower()).strip('_')
    model_clean = re.sub(r'[^a-z0-9]+', '_', model_name.lower()).strip('_')
    return f"{make_clean}_{model_clean}"

# Add model_id to each review
for review in complete_reviews:
    review['model_id'] = generate_model_id(review['make_name'], review['model_name'])

print("✓ Generated model_id for all reviews")
print(f"Example: {complete_reviews[0]['make_name']} {complete_reviews[0]['model_name']} -> {complete_reviews[0]['model_id']}")

✓ Generated model_id for all reviews
Example: Maruti Suzuki Ciaz -> maruti_suzuki_ciaz


## Step 2: Generate Concise Summaries using OpenAI GPT-4

In [5]:
def generate_concise_summary(full_summary: str, make: str, model: str, variant: str = "", retry_count: int = 3) -> str:
    """
    Generate a concise 20-word summary using OpenAI GPT-4.
    Includes retry logic for API failures.
    """
    vehicle_name = f"{model} {variant}".strip() if variant else model
    
    prompt = f"""Write a crisp, exactly 20-word summary of the {make} {vehicle_name}.

Review:
{full_summary}

Requirements:
- Make it use-case focused (family, city driving, performance, long trips, etc.) and clearly reflect what the car and variant stand for.
- Highlight only the most important differentiating features if they are true key selling points.
- Avoid fluff, repetition, and generic phrases. Keep it sharp, informative, and user-focused.
- Do not begin the sentence with the car name.
- The summary MUST be exactly 20 words.

Provide ONLY the 20-word summary, nothing else."""
    
    for attempt in range(retry_count):
        try:
            response = client.chat.completions.create(
                model="gpt-4.1",
                messages=[
                    {"role": "system", "content": "You are an expert automotive reviewer who creates crisp, use-case focused summaries."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.7,
                max_tokens=300
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            if attempt < retry_count - 1:
                print(f"  Retry {attempt + 1}/{retry_count} for {make} {model}...")
                time.sleep(2 ** attempt)  # Exponential backoff
            else:
                print(f"  ✗ Failed to generate summary for {make} {model}: {str(e)}")
                return f"Summary generation failed for {make} {model}."
    
    return f"Summary generation failed for {make} {model}."

print("✓ Summary generation function ready")

✓ Summary generation function ready


In [6]:
# Generate summaries for all reviews
print(f"Starting summary generation for {len(complete_reviews)} reviews...\n")

for idx, review in enumerate(complete_reviews, 1):
    print(f"[{idx}/{len(complete_reviews)}] Processing: {review['make_name']} {review['model_name']}")
    
    # Generate concise summary
    model_summary = generate_concise_summary(
        review['summary'],
        review['make_name'],
        review['model_name']
    )
    
    review['model_summary'] = model_summary
    print(f"  ✓ Generated {len(model_summary)} character summary\n")
    
    # Rate limiting to avoid API throttling
    time.sleep(0.5)

print("\n✓ All summaries generated successfully!")

Starting summary generation for 266 reviews...

[1/266] Processing: Maruti Suzuki Ciaz
  ✓ Generated 143 character summary

[2/266] Processing: Rolls-Royce Cullinan
  ✓ Generated 154 character summary

[3/266] Processing: Lexus ES
  ✓ Generated 150 character summary

[4/266] Processing: Audi e-tron
  ✓ Generated 135 character summary

[5/266] Processing: BMW M8
  ✓ Generated 151 character summary

[6/266] Processing: Mahindra Marazzo
  ✓ Generated 132 character summary

[7/266] Processing: Audi A4
  ✓ Generated 154 character summary

[8/266] Processing: Maruti Suzuki Celerio
  ✓ Generated 138 character summary

[9/266] Processing: Land Rover Defender
  ✓ Generated 147 character summary

[10/266] Processing: Jaguar F-Pace
  ✓ Generated 157 character summary

[11/266] Processing: Porsche Macan
  ✓ Generated 143 character summary

[12/266] Processing: BMW iX
  ✓ Generated 147 character summary

[13/266] Processing: Mini Cooper SE
  ✓ Generated 124 character summary

[14/266] Processing: M

## Step 3: Save updated data to complete_review.json

In [7]:
# Backup original file
import shutil
shutil.copy('complete_review.json', 'complete_review.json.backup')
print("✓ Created backup: complete_review.json.backup")

# Save updated reviews with model_id and model_summary
with open('complete_review.json', 'w', encoding='utf-8') as f:
    json.dump(complete_reviews, f, indent=4, ensure_ascii=False)

print("✓ Saved updated data to complete_review.json")
print(f"  Added fields: model_id, model_summary to {len(complete_reviews)} reviews")

✓ Created backup: complete_review.json.backup
✓ Saved updated data to complete_review.json
  Added fields: model_id, model_summary to 266 reviews


## Step 4: Update final_data.json with model_summary

In [12]:
# Create a lookup dictionary for quick access
summary_lookup = {
    review['model_id']: review['model_summary']
    for review in complete_reviews
}

print(f"✓ Created lookup dictionary with {len(summary_lookup)} model summaries")

✓ Created lookup dictionary with 266 model summaries


In [13]:
# Update final_data.json records
print(f"Updating {len(final_data)} records in final_data.json...\n")

matched_count = 0
unmatched_count = 0

for idx, record in enumerate(final_data, 1):
    if idx % 10000 == 0:
        print(f"  Processed {idx}/{len(final_data)} records...")
    
    # Generate model_id from make and model fields
    if 'make' in record and 'model' in record:
        # Convert display_make and display_model to match the format used in complete_review.json
        make_for_lookup = record.get('display_make', record['make'])
        model_for_lookup = record.get('display_model', record['model'])
        
        # Generate model_id using the same logic as in complete_review.json
        model_id = generate_model_id(make_for_lookup, model_for_lookup)
        
        # Look up corresponding summary
        if model_id in summary_lookup:
            record['model_summary'] = summary_lookup[model_id]
            matched_count += 1
        else:
            unmatched_count += 1
    else:
        unmatched_count += 1

print(f"\n✓ Update complete:")
print(f"  Matched and updated: {matched_count} records")
print(f"  Unmatched: {unmatched_count} records")

Updating 1974 records in final_data.json...


✓ Update complete:
  Matched and updated: 1972 records
  Unmatched: 2 records


In [14]:
# Backup original final_data.json
shutil.copy('final_data.json', 'final_data.json.backup')
print("✓ Created backup: final_data.json.backup")

# Save updated final_data.json
with open('final_data.json', 'w', encoding='utf-8') as f:
    json.dump(final_data, f, indent=2, ensure_ascii=False)

print("✓ Saved updated data to final_data.json")
print(f"  Added model_summary to {matched_count} matching records")

✓ Created backup: final_data.json.backup
✓ Saved updated data to final_data.json
  Added model_summary to 1972 matching records


## Summary Report

In [13]:
print("="*60)
print("SUMMARY REPORT")
print("="*60)
print(f"\nReviews processed: {len(complete_reviews)}")
print(f"Model IDs generated: {len(summary_lookup)}")
print(f"Final data records updated: {matched_count}")
print(f"\nFiles updated:")
print(f"  - complete_review.json (added model_id and model_summary)")
print(f"  - final_data.json (added model_summary to matching records)")
print(f"\nBackup files created:")
print(f"  - complete_review.json.backup")
print(f"  - final_data.json.backup")
print("\n✓ All tasks completed successfully!")
print("="*60)

SUMMARY REPORT

Reviews processed: 266
Model IDs generated: 266
Final data records updated: 1972

Files updated:
  - complete_review.json (added model_id and model_summary)
  - final_data.json (added model_summary to matching records)

Backup files created:
  - complete_review.json.backup
  - final_data.json.backup

✓ All tasks completed successfully!


In [15]:
# Display sample results
print("\nSample Generated Summaries:\n")
for i in range(min(3, len(complete_reviews))):
    review = complete_reviews[i]
    print(f"Make/Model: {review['make_name']} {review['model_name']}")
    print(f"Model ID: {review['model_id']}")
    print(f"Summary: {review['model_summary']}")
    print("-" * 60 + "\n")


Sample Generated Summaries:

Make/Model: Maruti Suzuki Ciaz
Model ID: maruti_suzuki_ciaz
Summary: Spacious, fuel-efficient sedan ideal for families and city driving; delivers comfort, huge boot, easy maintenance, but lacks advanced features.
------------------------------------------------------------

Make/Model: Rolls-Royce Cullinan
Model ID: rolls_royce_cullinan
Summary: Unparalleled comfort, opulent cabin, and bespoke luxury make this the ultimate SUV for affluent families prioritizing exclusivity over cost or efficiency.
------------------------------------------------------------

Make/Model: Lexus ES
Model ID: lexus_es
Summary: Ideal for comfort-focused city families, the ES 300h delivers plush refinement, class-leading efficiency, hybrid smoothness, and standout reliability.
------------------------------------------------------------

